# Geetabitan PDF OCR and search index

This Colab workflow OCRs the supplied Bengali PDF with Tesseract's Bengali model, saves the full page transcription privately in Google Drive, and builds a compact search enrichment keyed to the existing Geetabitan catalogue. The enrichment contains song source URLs, PDF page numbers, and Romanized Bengali search tokens; it does not contain full OCR lyrics.

**Keep the PDF in your own Drive.** If the PDF is only in Downloads, the upload cell below sends it to the Colab runtime and then copies it into your Drive output folder. Run each section in order. The 2,060-page document may take a while.


In [ ]:
# Install OCR and PDF tools in this Colab runtime.
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ben poppler-utils
!pip -q install pytesseract pdf2image pypdf indic-transliteration


In [ ]:
from pathlib import Path
import json, re, shutil, time
from google.colab import drive, files
from pypdf import PdfReader
from pdf2image import convert_from_path
from PIL import ImageOps
import pytesseract

drive.mount('/content/drive')
OUT_DIR = Path('/content/drive/MyDrive/TagoreGPT_OCR')
OUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = Path('/content/drive/MyDrive/gitobitan_all_bng.pdf')
if not PDF_PATH.exists():
    print('Choose gitobitan_all_bng.pdf from your computer. It will be copied to your Drive output folder.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No PDF was uploaded.')
    uploaded_name = next(iter(uploaded))
    runtime_pdf = Path('/content') / uploaded_name
    PDF_PATH = OUT_DIR / 'gitobitan_all_bng.pdf'
    shutil.copy2(runtime_pdf, PDF_PATH)
print('PDF:', PDF_PATH)
print('Pages:', len(PdfReader(str(PDF_PATH)).pages))


In [ ]:
# OCR all pages in chunks so each render batch fits in memory.
from concurrent.futures import ThreadPoolExecutor
from pytesseract import image_to_string

PAGE_COUNT = len(PdfReader(str(PDF_PATH)).pages)
OCR_JSONL = OUT_DIR / 'gitobitan_ocr_pages.jsonl'
DPI = 220
CHUNK_SIZE = 20

def ocr_page(item):
    page_number, image = item
    image = ImageOps.grayscale(image)
    text = image_to_string(image, lang='ben+eng', config='--psm 6')
    return {'pdf_page': page_number, 'text_bengali': text.strip()}

started = time.time()
with OCR_JSONL.open('w', encoding='utf-8') as out:
    for first in range(1, PAGE_COUNT + 1, CHUNK_SIZE):
        last = min(first + CHUNK_SIZE - 1, PAGE_COUNT)
        images = convert_from_path(str(PDF_PATH), dpi=DPI, first_page=first, last_page=last, fmt='png', thread_count=2)
        with ThreadPoolExecutor(max_workers=2) as pool:
            results = pool.map(ocr_page, ((first + offset, image) for offset, image in enumerate(images)))
            for record in results:
                out.write(json.dumps(record, ensure_ascii=False) + '\n')
        del images
        print(f'OCR complete through page {last:,}/{PAGE_COUNT:,} ({(time.time()-started)/60:.1f} min)')
print('Full OCR saved privately to:', OCR_JSONL)


## Build the compact index

This next cell transliterates OCR text to ITRANS, compares those tokens to the website's romanized lyric terms, and keeps only strong page matches. Review the printed match count and a few examples before using the exported index.


In [ ]:
from indic_transliteration import sanscript
import requests
from collections import Counter, defaultdict
from math import log

CATALOGUE_URL = 'https://malhar02.github.io/tagore-gpt/songs.json'
response = requests.get(CATALOGUE_URL, timeout=60)
response.raise_for_status()
songs = response.json()
print('Website catalogue rows:', len(songs))

STOP = set('ami tumi tomar amar tar a je se na go o re ki keno kotha more tore priyo'.split())
def roman_tokens(bengali_text):
    roman = sanscript.transliterate(bengali_text, sanscript.BENGALI, sanscript.ITRANS)
    roman = roman.lower().replace('ā','a').replace('ī','i').replace('ū','u')
    roman = re.sub(r'[^a-z]+', ' ', roman)
    return {w for w in roman.split() if len(w) >= 3 and w not in STOP}

pages = {}
with OCR_JSONL.open(encoding='utf-8') as source:
    for line in source:
        record = json.loads(line)
        if record.get('text_bengali'):
            pages[int(record['pdf_page'])] = roman_tokens(record['text_bengali'])
print('Pages with OCR text:', len(pages))

page_frequency = Counter(term for terms in pages.values() for term in terms)
page_inverted = defaultdict(list)
for page, terms in pages.items():
    for term in terms:
        page_inverted[term].append(page)
idf = {term: log(1 + len(pages)/(frequency + 1)) for term, frequency in page_frequency.items()}

enrichment = []
for song in songs:
    terms = {str(t).lower() for t in song.get('search_terms', []) if len(str(t)) >= 3 and str(t).lower() not in STOP}
    if not terms:
        continue
    weights = {term: idf.get(term, 0.0) for term in terms}
    total_weight = sum(weights.values())
    if total_weight <= 0:
        continue
    page_scores = defaultdict(float)
    page_hits = defaultdict(list)
    for term in terms:
        for page in page_inverted.get(term, []):
            page_scores[page] += weights[term]
            page_hits[page].append(term)
    if not page_scores:
        continue
    best_page = max(page_scores, key=page_scores.get)
    hits = sorted(set(page_hits[best_page]))
    coverage = page_scores[best_page] / total_weight
    if len(hits) >= 3 and coverage >= 0.12:
        enrichment.append({
            'source_url': song.get('source_url'),
            'pdf_page': best_page,
            'pdf_search_terms': hits,
            'pdf_match_score': round(coverage, 3),
        })

INDEX_PATH = OUT_DIR / 'pdf_search_index.json'
INDEX_PATH.write_text(json.dumps(enrichment, ensure_ascii=False, indent=2), encoding='utf-8')
print('Matched songs:', len(enrichment), '/', len(songs))
print('Index saved to:', INDEX_PATH)
print('Sample:', json.dumps(enrichment[:3], ensure_ascii=False, indent=2))


In [ ]:
# Download the compact index so it can be added to the public app repository.
# The full OCR transcript remains in your Google Drive.
files.download(str(INDEX_PATH))
